# 三向人工智能辩论

三位法学硕士之间的结构化辩论，每位法学硕士都有独特的个性：

- **Alex (GPT-4.1-mini)** — 主持人。介绍主题，提出探究性问题，让事情步入正轨。
- **约翰（双子座）** — 乐观主义者。处处看到机遇，看好未来。
- **彼得（克劳德）** — 怀疑论者。质疑假设，唱反调。

每个模型都会收到一个定义其角色的系统提示，以及一个包含完整信息的用户提示
到目前为止的谈话记录。

In [ ]:
# 进口

from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv
import os

In [ ]:
# 从 .env 文件加载环境变量

load_dotenv(override=True)

### 客户

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
openai_client = OpenAI()

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

### 模特与人物

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
ALEX_MODEL = "gpt-4.1-mini"
JOHN_MODEL = "google/gemini-2.5-flash-lite"
PETER_MODEL = "anthropic/claude-3.5-haiku"

In [ ]:
# 系统提示

alex_system = """You are Alex, the moderator of a debate.
You introduce the topic, ask probing follow-up questions, challenge weak arguments,
and keep the discussion focused. You are fair but push both sides to go deeper.
Keep your responses concise — 2 to 4 sentences max.
You are in a debate with John and Peter."""

john_system = """You are John, a debater who is optimistic and forward-thinking.
You see opportunity and potential in new developments. You back your points
with practical examples and real-world impact.
Keep your responses concise — 3 to 5 sentences max.
You are in a debate with Peter. Alex is the moderator."""

peter_system = """You are Peter, a debater who is skeptical and analytical.
You question assumptions, point out risks, and demand evidence.
You're not negative — you just want to stress-test ideas before buying in.
Keep your responses concise — 3 to 5 sentences max.
You are in a debate with John. Alex is the moderator."""

### 主题

In [ ]:
# 改变这个来辩论不同的主题

topic = "Why does it matter right now to learn LLM engineering? Is this the right time, or is it too early / too late?"

In [ ]:
# 共享记录——单一事实来源

transcript = []

In [ ]:
# 将成绩单格式化为可读字符串

def format_transcript():
    if not transcript:
        return "(No conversation yet)"
    lines = []
    for entry in transcript:
        lines.append(f"{entry['speaker']}: {entry['text']}")
    return "\n\n".join(lines)

In [ ]:
# 亚历克斯·卡尔

def call_alex(instruction):
    user_prompt = f"""{instruction}

The conversation so far:
{format_transcript()} 

Respond as Alex the moderator."""
    
    response = openai_client.chat.completions.create(
        model=ALEX_MODEL,
        messages=[
            {"role": "system", "content": alex_system},
            {"role": "user", "content": user_prompt}
        ],
    )

    reply = response.choices[0].message.content
    transcript.append({"speaker": "Alex (Moderator)", "text": reply})
    return reply

In [ ]:
# 打电话给约翰

def call_john():
    user_prompt = f"""You are John in a debate moderated by Alex.
The conversation so far:
{format_transcript()}
Respond to what just been said, Stay in character as the optimistic and forward-thinking John."""
    
    response = openrouter_client.chat.completions.create(
        model=JOHN_MODEL,
        messages=[
            {"role": "system", "content": john_system},
            {"role": "user", "content": user_prompt}
        ],
    )
    reply = response.choices[0].message.content
    transcript.append({"speaker": "John (Optimist)", "text": reply})
    return reply

In [ ]:
# 打电话给彼得

def call_peter():
    user_prompt = f"""You are Peter in a debate moderated by Alex.

The conversation so far:
{format_transcript()}

Respond to what was just said. Stay in character as the skeptic."""

    response = openrouter_client.chat.completions.create(
        model=PETER_MODEL,
        messages=[
            {"role": "system", "content": peter_system},
            {"role": "user", "content": user_prompt}
        ]
    )
    reply = response.choices[0].message.content
    transcript.append({"speaker": "Peter (Skeptic)", "text": reply})
    return reply

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def show(speaker, text):
    display(Markdown(f"### {speaker}\n\n{text}\n\n---"))

### 辩论

In [ ]:
# 第 0 轮 — Alex 介绍主题

transcript = []  # reset

opening = call_alex(f"Introduce this debate topic to John and Peter: {topic}")
show("Alex (Moderator)", opening)

In [ ]:
# 第 1 轮至第 5 轮

for i in range(1, 6):
    display(Markdown(f"## Round {i}"))

    # 约翰回应
    john_reply = call_john()
    show("John (Optimist)", john_reply)

    # 彼得回应
    peter_reply = call_peter()
    show("Peter (Skeptic)", peter_reply)

    # 亚历克斯温和派
    alex_reply = call_alex("Ask a follow-up question or challenge one of them. Push the debate deeper.")
    show("Alex (Moderator)", alex_reply)

In [ ]:
# 亚历克斯致闭幕词

closing = call_alex("Summarize the key points from both sides and give your closing remarks to end the debate.")
show("Alex (Moderator)", closing)

### 完整成绩单

In [ ]:
print(format_transcript())